In [4]:
print("OK")

OK


In [5]:
%pwd

'd:\\Ai-project\\medicalChatbot\\research'

In [14]:
import os
os.chdir("../")

In [7]:
%pwd

'd:\\Ai-project\\medicalChatbot'

In [17]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [18]:
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)
    
    documents=loader.load()

    return documents

In [19]:
extracted_data=load_pdf_file(data='Data/')

In [22]:
#extracted_data

In [23]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [24]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 40000


In [16]:
#text_chunks

In [3]:
from langchain.embeddings import HuggingFaceEmbeddings

In [1]:
def download_hugging_face_embedding():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLm-L6-v2')
    return embeddings

In [5]:
def download_hugging_face_embedding():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [10]:
embeddings= download_hugging_face_embedding()

In [11]:
query_result= embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [20]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "medicalbot",
    "metric": "cosine",
    "host": "medicalbot-g7o896m.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [26]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [27]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [28]:
docsearch

In [29]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [30]:
retriever_docs = retriever.invoke("What is Acne?")

In [31]:
retriever_docs

[Document(id='8f5bc127-2ffe-4e8f-b926-f24b10a9af09', metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2006-10-16T22:03:45+02:00', 'page': 55.0, 'page_label': '26', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 4505.0}, page_content='Researchers, Inc. Reproduced by permission.)\n26 GALE ENCYCLOPEDIA OF MEDICINE\nAcne'),
 Document(id='2e27a032-7b44-40b9-aef3-dd6f24270107', metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2006-10-16T22:03:45+02:00', 'page': 55.0, 'page_label': '26', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 4505.0}, page_content='Sebaceous follicles— A structure found within the\nskin that houses the oil-producing glands and hair\nfollicles, where pimples form.\nSebum— An oily skin moisturizer produced by\nsebaceous glands.\nTretinoin— A drug that works by increasing the\nturnover 

In [35]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import os

# Load .env reliably (works from notebooks even if CWD changes)
load_dotenv(find_dotenv(usecwd=True), override=False)

# Sanity-check
openai_key = os.getenv("OPENAI_API_KEY")
pinecone_key = os.getenv("PINECONE_API_KEY")

assert openai_key, "OPENAI_API_KEY missing. Check your .env and reload kernel."
assert pinecone_key, "PINECONE_API_KEY missing. Check your .env and reload kernel."

print("Keys loaded OK ✅")


Keys loaded OK ✅


In [36]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

key = os.getenv("OPENAI_API_KEY")
print("Exists?", bool(key))
if key:
    print("Looks like an API key?", key.startswith("sk-"))
    print("Masked:", key[:7] + "..." + key[-4:])


Exists? True
Looks like an API key? True
Masked: sk-proj...mo8A


In [37]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.4, max_tokens=500)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "z
    "answer concise. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [39]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [40]:
response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])



Acne is a skin disorder that occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. It is characterized by the presence of pimples on the face, chest, and back. Tretinoin is a drug that can help treat acne by increasing the turnover of skin cells.


In [41]:
response = rag_chain.invoke({"input": "What is ice cream?"})
print(response["answer"])



Ice cream is a frozen dessert made from cream, sugar, and flavorings. It is typically served in a scoop or cone and can come in a variety of flavors. It is a popular treat enjoyed by people of all ages.


In [42]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])



Stats, or statistics, is a branch of mathematics that deals with the collection, analysis, interpretation, presentation, and organization of data. It involves using mathematical and computational methods to gather, summarize, and analyze data in order to make informed decisions or draw conclusions about a population or phenomenon. It is used in various fields such as science, economics, psychology, and business to make sense of large amounts of data and make predictions or inferences based on that data.


In [43]:
response = rag_chain.invoke({"input": "What is Lymph node biopsy?"})
print(response["answer"])



Lymph node biopsy is a medical procedure in which a sample of tissue is taken from a lymph node to check for the presence of cancer. It can be done either through a needle or through surgery, depending on the location of the lymph node. The procedure is used to determine if there is cancer within the lymph node.


In [44]:
response = rag_chain.invoke({"input": "awpda[d[ajaf8f8]]"})
print(response["answer"])



I'm sorry, I don't understand the question. Can you please rephrase it?
